# Ticket 2: load stg_gl verbatim from source

What `src/load_stg.py` checks before trusting the load, run here to look
at the source directly.

In [1]:
import duckdb # type: ignore

con = duckdb.connect()
SOURCE = "../dataset/journal_entries.parquet"
con.execute(f"SELECT COUNT(*) FROM read_parquet('{SOURCE}')").fetchall()

[(648801,)]

**648,801 rows** in the source parquet — the number `stg_gl` must match
exactly. `src/load_stg.py` asserts this on every run.

In [2]:
# schema: name + type, column by column
con.execute(f"DESCRIBE SELECT * FROM read_parquet('{SOURCE}')").fetchall()

[('document_id', 'VARCHAR', 'YES', None, None, None),
 ('company_code', 'BIGINT', 'YES', None, None, None),
 ('fiscal_year', 'BIGINT', 'YES', None, None, None),
 ('fiscal_period', 'BIGINT', 'YES', None, None, None),
 ('posting_date', 'VARCHAR', 'YES', None, None, None),
 ('document_date', 'VARCHAR', 'YES', None, None, None),
 ('document_type', 'VARCHAR', 'YES', None, None, None),
 ('currency', 'VARCHAR', 'YES', None, None, None),
 ('exchange_rate', 'DOUBLE', 'YES', None, None, None),
 ('reference', 'VARCHAR', 'YES', None, None, None),
 ('header_text', 'VARCHAR', 'YES', None, None, None),
 ('created_by', 'VARCHAR', 'YES', None, None, None),
 ('source', 'VARCHAR', 'YES', None, None, None),
 ('business_process', 'VARCHAR', 'YES', None, None, None),
 ('ledger', 'VARCHAR', 'YES', None, None, None),
 ('is_fraud', 'BOOLEAN', 'YES', None, None, None),
 ('is_anomaly', 'BOOLEAN', 'YES', None, None, None),
 ('line_number', 'BIGINT', 'YES', None, None, None),
 ('gl_account', 'BIGINT', 'YES', None,

**49 columns.** `src/load_stg.py` compares this list against `stg_gl`'s
schema name-for-name and type-for-type after load — a rename or an
implicit cast would fail the check, since `stg_gl` is meant to be a
faithful copy, not a transform.

In [3]:
# shape of the dataset: distinct documents, companies, fiscal year range
con.execute(f"""
    SELECT COUNT(DISTINCT document_id), COUNT(DISTINCT company_code),
           MIN(fiscal_year), MAX(fiscal_year)
    FROM read_parquet('{SOURCE}')
""").fetchall()

[(174944, 4, 2024, 2025)]

**174,944 distinct documents, 4 companies, FY2024-2025.** The document
count became the anchor value in `src/checks.py`
(`stg_gl_document_count_stable`) — a change here on a future dataset
refresh should fail loud, not shift silently.

## What I've got

The source parquet matches `stg_gl` row for row, column for column: same
row count, same schema, same document/company/year shape. `stg_gl` is a
faithful copy, not a transform — this is what proves it.